# NeoOLAF native DocRED layer ablation — Skai TV (corrected profile/RAG)

This notebook diagnoses the first DocRED document (**Skai TV**) layer by layer.

The corrected experiment addresses two problems revealed by the first run without
changing anything under `src/neoolaf`:

1. Layer 3 now uses NeoOLAF's existing profile-selected role-based strategy, so
   distinct named mentions keep their source labels instead of being merged into
   generic candidates such as `entity` or `location`.
2. The notebook ontology adapter retrieves classes and properties separately.
   This guarantees that relation-oriented prompts actually receive DocRED
   property descriptions instead of a top-k list containing only classes.

Scientific constraints remain unchanged:

- the complete 13-component NeoOLAF pipeline is run;
- no additional direct DocRED extraction call is used;
- no source/gold entity anchoring is used;
- no gold relation hints, closure rules, or benchmark-specific relation invention are used;
- gold entities and relations are loaded only after execution;
- every layer saves restartable state, compact outputs, prompts, metadata, logs,
  errors, raw LLM responses, and ontology retrieval records.

The notebook performs a **cumulative layer analysis** from Layer 0 through Layer
12. It is not a leave-one-out ablation.

In [ ]:

from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "src/neoolaf").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the NeoOLAF repository.")


PROJECT_ROOT = find_project_root()
NOTEBOOK_DIR = PROJECT_ROOT / "examples/RAGTreeDatasets"
TOOLS_DIR = NOTEBOOK_DIR / "tools"
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

from docred_native_ablation import (
    analyze_run,
    load_layer_states,
    read_json,
    read_jsonl,
    run_native_pipeline,
)

print("PROJECT_ROOT =", PROJECT_ROOT)


## 1. Experiment configuration

In [ ]:

# Primary native experiment.
INPUT_JSONL = NOTEBOOK_DIR / "data/docred_skai_tv_input.jsonl"
GOLD_JSONL = NOTEBOOK_DIR / "data/docred_skai_tv_gold.jsonl"

# The original ontology is preserved verbatim. The compatible copy adds only
# owl:ObjectProperty type assertions because NeoOLAF's current loader does not
# index predicates declared solely as rdf:Property.
ONTOLOGY_ORIGINAL = NOTEBOOK_DIR / "ontology/docred_redocred_original.ttl"
ONTOLOGY_PATH = NOTEBOOK_DIR / "ontology/docred_redocred_neoolaf_compatible.ttl"
RELATION_CATALOG = NOTEBOOK_DIR / "ontology/docred_relation_catalog.json"
RELATION_ALIASES = NOTEBOOK_DIR / "ontology/docred_relation_aliases.json"

PROFILE_PATH = NOTEBOOK_DIR / "configs/docred_profile_native_ablation.json"
GUIDANCE_PATH = NOTEBOOK_DIR / "configs/guidance_docred_native_ablation.json"

RUNS_ROOT = NOTEBOOK_DIR / "runs/docred_native_layer_ablation"
RUN_DIR = RUNS_ROOT / "skai_tv_balanced_v2"

# LLM configuration. The API key is read only from the environment and is never printed.
OPENROUTER_HOST = "https://openrouter.ai/api/v1"
MODEL_NAME = "openai/gpt-oss-20b"
API_KEY_ENV = "OPENROUTER" + "_API_KEY"
API_KEY = os.environ.get(API_KEY_ENV, "")

# Concurrency is useful for Layers 2 and 3. Layer 1 has one whole-document chunk.
WORKERS = 4
MAX_TOKENS = 8192
REQUEST_TIMEOUT = 600
REASONING_EFFORT = "minimal"

# Keep True for the first execution. Set False to inspect an existing run without spending tokens.
RUN_PIPELINE = True

print("Input:", INPUT_JSONL)
print("Profile:", PROFILE_PATH)
print("Guidance:", GUIDANCE_PATH)
print("Ontology:", ONTOLOGY_PATH)
print("Run dir:", RUN_DIR)
print("Model:", MODEL_NAME)
print("API key available:", bool(API_KEY))


## 2. Preflight checks and previous-v6 context

In [ ]:

required = [
    INPUT_JSONL, GOLD_JSONL, ONTOLOGY_ORIGINAL, ONTOLOGY_PATH,
    RELATION_CATALOG, RELATION_ALIASES, PROFILE_PATH, GUIDANCE_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

input_rows = read_jsonl(INPUT_JSONL)
gold_rows = read_jsonl(GOLD_JSONL)
assert len(input_rows) == 1
assert len(gold_rows) == 1
assert "entities" not in input_rows[0] and "relations" not in input_rows[0]
assert input_rows[0]["document_id"] == gold_rows[0]["document_id"]

catalog = read_json(RELATION_CATALOG)
profile = read_json(PROFILE_PATH)
guidance = read_json(GUIDANCE_PATH)
baseline_v6 = read_json(NOTEBOOK_DIR / "reference/v6_calibrated_baseline_summary.json")

print("Document:", input_rows[0]["document_id"], "-", input_rows[0]["title"])
print("Document characters:", len(input_rows[0]["text"]))
print("Ontology classes:", catalog["class_count"])
print("Ontology properties:", catalog["property_count"])
print("Active profile:", profile["profile_name"])
print("Layer 3 strategy:", profile["layers"]["layer03_candidate_typing_resolution"]["strategy"])
print("RAG backend:", profile["rag"]["backend"])
print("Guidance priority relations:", len(guidance["priority_relations"]))
print()
print("Previous v6 five-document benchmark-facing score:")
print({
    "precision": round(baseline_v6["precision"], 4),
    "recall": round(baseline_v6["recall"], 4),
    "f1": round(baseline_v6["f1"], 4),
})
print("Important:", baseline_v6["important_interpretation"])

display(Markdown(
    "**Ontology compatibility note.** The supplied ontology declares 96 predicates as "
    "`rdf:Property`. The current `SeedOntologyLoader` indexes only "
    "`owl:ObjectProperty`/`owl:DatatypeProperty`, so the notebook uses a semantically "
    "equivalent copy with additional `owl:ObjectProperty` type assertions. The original "
    "ontology is preserved unchanged beside it."
))


## 3. Run the full native pipeline

In [ ]:

if RUN_PIPELINE:
    if not API_KEY:
        raise RuntimeError(
            "Set OPENROUTER_API_KEY in the environment before running this cell. "
            "Do not paste the key into the notebook."
        )

    final_state = run_native_pipeline(
        project_root=PROJECT_ROOT,
        input_jsonl=INPUT_JSONL,
        ontology_path=ONTOLOGY_PATH,
        profile_path=PROFILE_PATH,
        guidance_path=GUIDANCE_PATH,
        run_dir=RUN_DIR,
        model_name=MODEL_NAME,
        api_key=API_KEY,
        host=OPENROUTER_HOST,
        workers=WORKERS,
        max_tokens=MAX_TOKENS,
        request_timeout=REQUEST_TIMEOUT,
        reasoning_effort=REASONING_EFFORT,
        verbose=True,
    )
    print("Full native run completed.")
else:
    print("RUN_PIPELINE=False: reusing artifacts from", RUN_DIR)


### Saved runtime evidence

After the previous cell, inspect:

- `run_manifest.json`
- `run_logs/console.log`
- `run_logs/llm_calls.jsonl`
- `run_logs/llm_errors.jsonl`
- `run_logs/pipeline_errors.jsonl`
- `run_logs/ontology_retrieval.jsonl`
- `<layer>/state.json`
- `<layer>/output.json`
- `<layer>/metadata.json`
- `<layer>/prompts/`
- `<layer>/prompt_stats.json`
- `exports/`

The ontology log now includes `result_types` and `property_ids`. For Layer 4,
each retrieval should contain DocRED properties as well as classes.

Raw responses are saved without API credentials under `run_logs/responses/`.

## 4. Cumulative layer analysis

In [ ]:

summary = analyze_run(
    run_dir=RUN_DIR,
    gold_jsonl=GOLD_JSONL,
    catalog_path=RELATION_CATALOG,
    aliases_path=RELATION_ALIASES,
)

layer_df = pd.DataFrame(summary["layer_summary"])
display(layer_df)

print("Strict native DocRED evaluation:")
display(pd.DataFrame([{
    key: summary["strict_evaluation"][key]
    for key in ["predicted", "gold", "true_positive", "false_positive",
                "false_negative", "precision", "recall", "f1"]
}]))
print("Failure counts:", summary["failure_counts"])


## 5. Trace each gold relation from Layer 1 to its first loss

In [ ]:

trace_path = RUN_DIR / "analysis/gold_relation_trace.csv"
trace_df = pd.read_csv(trace_path)
display(trace_df)

display(
    trace_df.groupby("first_failure", dropna=False)
    .size()
    .reset_index(name="gold_relations")
    .sort_values("gold_relations", ascending=False)
)


## 6. Inspect native candidates and assertions around the expected bottleneck

In [ ]:

states = {index: state for index, _, state in load_layer_states(RUN_DIR)}

layer1 = states.get(1)
layer3 = states.get(3)
layer4 = states.get(4)
layer5 = states.get(5)

if layer1:
    display(pd.DataFrame([
        {"expr_id": x.expr_id, "text": x.text, "label": x.label, "justification": x.justification}
        for x in layer1.linguistic_expressions
    ]))

if layer3:
    node_rows = []
    for family, candidates in [
        ("entity", layer3.entity_candidates),
        ("event", layer3.event_candidates),
        ("attribute", layer3.attribute_candidates),
    ]:
        for candidate in candidates:
            node_rows.append({
                "family": family,
                "candidate_id": candidate.candidate_id,
                "canonical_label": candidate.canonical_label,
                "mentions": [m.text for m in candidate.mentions],
                "ontology_hints": candidate.ontology_hints,
                "confidence": candidate.confidence,
            })
    print("Layer 3 node candidates")
    display(pd.DataFrame(node_rows))

    print("Layer 3 relation candidates")
    display(pd.DataFrame([
        {
            "candidate_id": candidate.candidate_id,
            "canonical_label": candidate.canonical_label,
            "mentions": [m.text for m in candidate.mentions],
            "ontology_hints": candidate.ontology_hints,
            "confidence": candidate.confidence,
        }
        for candidate in layer3.relation_candidates
    ]))

if layer4:
    print("Layer 4 candidate relation assertions")
    display(pd.DataFrame([
        {
            "assertion_id": x.assertion_id,
            "source": x.source_candidate_label,
            "relation": x.relation_label,
            "target": x.target_candidate_label,
            "confidence": x.confidence,
            "justification": x.justification,
        }
        for x in layer4.candidate_relation_assertions
    ]))

if layer5:
    print("Layer 5 candidate triples")
    display(pd.DataFrame([
        {
            "triple_id": x.triple_id,
            "subject": x.subject_label,
            "predicate": x.predicate_label,
            "object": x.object_label,
            "confidence": x.confidence,
            "justification": x.justification,
        }
        for x in layer5.candidate_triples
    ]))


## 7. Ontology retrieval and LLM-call diagnostics

In [ ]:

retrieval_path = RUN_DIR / "run_logs/ontology_retrieval.jsonl"
llm_calls_path = RUN_DIR / "run_logs/llm_calls.jsonl"
llm_errors_path = RUN_DIR / "run_logs/llm_errors.jsonl"

if retrieval_path.is_file():
    retrieval_df = pd.DataFrame(read_jsonl(retrieval_path))
    display(retrieval_df)
    print("Retrievals by layer:")
    display(retrieval_df.groupby("layer_name").size().reset_index(name="retrieval_calls"))

if llm_calls_path.is_file():
    llm_df = pd.DataFrame(read_jsonl(llm_calls_path))
    display(llm_df)
    print("LLM calls:", len(llm_df))
    print("Total recorded runtime (not wall-clock due to parallelism):", llm_df["elapsed_seconds"].sum())

if llm_errors_path.is_file():
    display(pd.DataFrame(read_jsonl(llm_errors_path)))
else:
    print("No logged LLM errors.")



## 8. Optional recall-oriented rerun

The files `docred_profile_native_ablation_relaxed.json` and
`guidance_docred_native_ablation_relaxed.json` are included for a controlled
secondary run. They lower candidate-promotion thresholds but keep:

- the same full NeoOLAF pipeline;
- the same ontology;
- the same strict evaluator;
- no direct extraction call;
- no source-entity anchoring;
- no closure rules.

Run this only after inspecting the balanced trace. Use a distinct output folder,
such as `runs/docred_native_layer_ablation/skai_tv_relaxed`, so the two runs remain
fully comparable.


In [ ]:

RUN_RELAXED_VARIANT = False

if RUN_RELAXED_VARIANT:
    relaxed_state = run_native_pipeline(
        project_root=PROJECT_ROOT,
        input_jsonl=INPUT_JSONL,
        ontology_path=ONTOLOGY_PATH,
        profile_path=NOTEBOOK_DIR / "configs/docred_profile_native_ablation_relaxed.json",
        guidance_path=NOTEBOOK_DIR / "configs/guidance_docred_native_ablation_relaxed.json",
        run_dir=RUNS_ROOT / "skai_tv_relaxed",
        model_name=MODEL_NAME,
        api_key=API_KEY,
        host=OPENROUTER_HOST,
        workers=WORKERS,
        max_tokens=MAX_TOKENS,
        request_timeout=REQUEST_TIMEOUT,
        reasoning_effort=REASONING_EFFORT,
        verbose=True,
    )
    relaxed_summary = analyze_run(
        run_dir=RUNS_ROOT / "skai_tv_relaxed",
        gold_jsonl=GOLD_JSONL,
        catalog_path=RELATION_CATALOG,
        aliases_path=RELATION_ALIASES,
    )
    display(pd.DataFrame([
        {"variant": "balanced", **{k: summary["strict_evaluation"][k] for k in ["precision", "recall", "f1"]}},
        {"variant": "relaxed", **{k: relaxed_summary["strict_evaluation"][k] for k in ["precision", "recall", "f1"]}},
    ]))
else:
    print("Relaxed variant disabled.")
